# Pipelines

This example ports a trained `Pipeline` of scikit-learn with
[sklearn-porter](https://github.com/nok/sklearn-porter). A pipeline of
supported transformers (the preprocessing) and a supported final estimator
(the prediction) is ported to one source file: the ported final estimator,
which transforms the features of each sample with the ported transformers
before its prediction. The functions `port`, `save`, `make` and `test`
accept pipelines too.

Making predictions with the ported pipeline requires a compiler or
interpreter of the target language, here `node` for JavaScript and the JDK
for Java.

## Train a pipeline

Load the data and train a pipeline:

In [1]:
from sklearn.datasets import load_iris
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X, y = load_iris(return_X_y=True)
pipe = make_pipeline(StandardScaler(), SVC())
pipe.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('standardscaler', ...), ('svc', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](3,)","[0,1,2]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,4
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


## Port the pipeline

**Step 1**: Port (transpile) the pipeline and get the source code:

In [2]:
from sklearn_porter import make, port, save, test

output = port(pipe, language='js', template='attached')

print(output)

/*
This file is generated by https://github.com/nok/sklearn-porter/

Estimator:
    SVC
    https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html

Environment:
    scikit-learn v1.9.1
    sklearn-porter v1.0.0

Usage:
    1. Execute a prediction:
        $ node Pipeline.js feature_1 ... feature_4
    2. Execute the predictions of several samples (one per line):
        $ node Pipeline.js < samples.txt
*/
var Pipeline = function(vectors, coeffs, inters, weights, kernel, gamma, coef0, degree) {
    this.nClasses = weights.length;
    this.vectors = vectors;
    this.coeffs = coeffs;
    this.inters = inters;
    this.weights = weights;
    this.kernel = kernel.toUpperCase();
    this.gamma = gamma;
    this.coef0 = coef0;
    this.degree = degree;

    // The labels of the classes:
    var LABELS = [0, 1, 2];

    // The power with an integer exponent (by repeated squaring) like libsvm:
    var powi = function(base, times) {
        var tmp = base, ret = 1., t;
        f

**Step 2**: Save the ported pipeline:

In [3]:
src_path = save(pipe, language='js', directory='/tmp')

print(src_path)

/tmp/Pipeline.js


**Step 3**: Make predictions with the ported pipeline, like
`pipe.predict(X[::15])`:

In [4]:
y_classes, _ = make(pipe, X[::15], language='js')

print(y_classes)

[0 0 0 0 1 1 1 2 2 2]


**Step 4**: Always check the ported pipeline: the score is the fraction of
predictions that match those of the original pipeline:

In [5]:
score = test(pipe, X[::15], language='js')

print(score)

1.0


## Use the ported pipeline

The generated file is used like a ported estimator: it reads the features
of one sample from its arguments or of several samples (one per line, the
features separated by spaces) from the standard input:

In [6]:
from pathlib import Path

lines = [' '.join(map(str, sample)) for sample in X[::30].tolist()]
Path('/tmp/samples.txt').write_text('\n'.join(lines) + '\n')

print(Path('/tmp/samples.txt').read_text())

5.1 3.5 1.4 0.2
4.8 3.1 1.6 0.2
5.0 2.0 3.5 1.0
5.5 2.6 4.4 1.2
6.9 3.2 5.7 2.3



In [7]:
!node /tmp/Pipeline.js < /tmp/samples.txt

Predicted class: 0
Predicted class: 0
Predicted class: 1
Predicted class: 1
Predicted class: 2


## Change the number of features

The transformers can change the number of features (e.g. a
`PolynomialFeatures` or a feature selector like `SelectKBest`); the ported
pipeline reads the features of the pipeline. The class `Porter` creates a
`Pipeline` for a pipeline:

In [8]:
from sklearn.feature_selection import SelectKBest
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures

from sklearn_porter import Porter

pipe = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False),
    StandardScaler(),
    SelectKBest(k=5),
    LogisticRegression(max_iter=1000),
)
pipe.fit(X, y)

porter = Porter(pipe, language='java')  # creates a `Pipeline`
y_classes, y_probas = porter.predict(X[::30])

print(y_classes)
print(y_probas)
print(porter.test(X[::15]))

[0 0 1 1 2]
[[9.81157335e-01 1.88425214e-02 1.43258115e-07]
 [9.75542808e-01 2.44569751e-02 2.16468061e-07]
 [1.26206923e-01 8.71991571e-01 1.80150682e-03]
 [1.20162841e-02 9.58914783e-01 2.90689328e-02]
 [2.54334036e-08 3.09787541e-03 9.96902099e-01]]


1.0


## Searches

A pipeline as the best estimator of a `GridSearchCV` or a
`RandomizedSearchCV` is ported too:

In [9]:
from sklearn.model_selection import GridSearchCV

search = GridSearchCV(
    make_pipeline(StandardScaler(), SVC()), {'svc__C': [0.1, 1.0, 10.0]}
)
search.fit(X, y)

print(test(search, X[::15], language='js'))

1.0


## Unsupported preprocessing steps

A pipeline with an unsupported transformer raises a `NotSupportedYetError`,
because the predictions of the ported final estimator alone would differ
from those of the pipeline. To port the final estimator anyway, pass
`ignore_preprocessing=True` and apply the preprocessing steps to the input
data of the ported estimator yourself, see the
[limitations](https://github.com/nok/sklearn-porter/blob/main/docs/limitations.md).